# RQ2 — 3-way end-to-end fixed-marginal pair pilot v2

Development seed 3 only. The default `screening` stage runs Resource and online Pure-SW together on T4×2 after one common Uniform-Pair warm-up. A later `complete` stage resumes the same v2 artifact, runs Uniform, then performs all diagnostics and finalization.

## Required inputs

- CIFAR-100 dataset containing `cifar-100-python/{train,test,meta}`.
- Frozen Gate-A output containing `gate_a_summary.json` (profiled FLOPs).
- Completed fresh seed-7 Pure-SW replication containing `fresh_seed7_puresw_replication_summary.json` with `decision=PASS`.
- Optional: an earlier **v2** `e2e_pairwise_pilot_v2` output for exact resume. v1 artifacts are intentionally rejected.
- Kaggle secret `github_token`.

The test split is never used. The seed-7 gate is only an authorization artifact; no seed-7 weights or accuracy enter this pilot.

In [ ]:
import os, subprocess, sys, json, time, zipfile, importlib, hashlib
from IPython.display import display
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count() == 2, f'Select T4 x2; detected {torch.cuda.device_count()} GPU(s)'
GPU_IDS = (0, 1)
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('GPUs:', [torch.cuda.get_device_name(i) for i in GPU_IDS])

## Resolve inputs and enforce the pre-registered gate

In [ ]:
import rq2_e2e_pairwise_pilot as pilot
pilot = importlib.reload(pilot)
INPUT_ROOT = Path('/kaggle/input')
DATASET_ROOT = pilot.find_cifar100_root(INPUT_ROOT)
GATE_A_SUMMARY = pilot.find_gate_a_summary(INPUT_ROOT)
SEED7_PASS = pilot.find_seed7_pass_summary(INPUT_ROOT)
print('CIFAR-100:', DATASET_ROOT)
print('Gate A:', GATE_A_SUMMARY)
print('Seed-7 authorization:', SEED7_PASS)
assert json.loads(SEED7_PASS.read_text())['decision'] == 'PASS'

## Select the stage, materialize optional progress, and freeze the pilot

Keep `RUN_STAGE='screening'` for the first 8–10 hour session. Later, attach its ZIP and change only this local control to `RUN_STAGE='complete'`; it is not a scientific protocol parameter.

In [ ]:
RUN_STAGE = 'screening'  # screening | complete
assert RUN_STAGE in {'screening', 'complete'}
ROOT = pilot.materialize_progress(
    INPUT_ROOT, '/kaggle/working/e2e_pairwise_pilot_v2',
    '/kaggle/working/materialized-e2e-pairwise-pilot-v2'
)
if RUN_STAGE == 'complete':
    assert (ROOT/'screening_summary.json').is_file(), 'Attach the completed v2 screening ZIP first'
CONFIG = pilot.load_config(PROJECT_ROOT/'configs/kaggle_s1_extension_100.yaml', DATASET_ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
resolved_path = ROOT/'resolved_config.yaml'
if not resolved_path.exists():
    import yaml
    resolved_path.write_text(yaml.safe_dump(CONFIG, sort_keys=False))
protocol = {
 'protocol_version':2,
 'status':'FROZEN_BEFORE_E2E_PAIRWISE_PILOT', 'seed':3,
 'methods':['uniform','resource','pure_sw'], 'common_warmup_epochs':10,
 'branch_epochs':[11,100], 'refresh_states':[10,20,30,40,50,60,70,80,90],
 'pair_marginal':1/7, 'pairs':91, 'subnets_per_batch':4,
 'schedule':'lr0.1_epochs1_50_then_optimizer_scheduler_reset_lr0.01_epochs51_100',
 'loss':'mean(CE_full, CE+KD_min, CE+KD_i, CE+KD_j)',
 'ht_or_importance_weighting':False, 'learned_hybrid':False,
 'equal_calibration_sweeps':True, 'test_used':False,
 'policy_geometry_source':'training_split_disjoint_from_bn_calibration',
 'policy_geometry_size':int(CONFIG['dataset']['feature_subset_size']),
 'validation_used_to_build_policy':False,
 'test_used_to_build_policy':False,
 'gate_a_sha256':hashlib.sha256(GATE_A_SUMMARY.read_bytes()).hexdigest(),
 'seed7_pass_sha256':hashlib.sha256(SEED7_PASS.read_bytes()).hexdigest(),
 'git_commit':GIT_COMMIT
}
protocol_path = ROOT/'frozen_protocol.json'
if protocol_path.exists():
    previous = json.loads(protocol_path.read_text())
    keys = [key for key in protocol if key != 'git_commit']
    assert all(previous.get(key) == protocol[key] for key in keys), 'Attached pilot violates frozen protocol'
else:
    protocol_path.write_text(json.dumps(protocol, indent=2)+'\n')
print(json.dumps(protocol, indent=2))

## Common epoch 1–10 Uniform-Pair warm-up

This runs once on GPU 0. Its epoch-10 checkpoint contains weights, optimizer, scheduler, data/augmentation RNG, and pair RNG shared by all branches.

In [ ]:
started = time.perf_counter()
common_checkpoint = pilot.train_common_warmup(CONFIG, ROOT)
print('Common checkpoint:', common_checkpoint)
print(f'Warm-up elapsed: {(time.perf_counter()-started)/3600:.2f} h')

## Train the selected branches on T4×2

The screening stage runs Resource and Pure-SW concurrently. The complete stage runs the remaining Uniform branch from the exact common epoch-10 checkpoint. Every branch is exactly resumable.

In [ ]:
import scripts.run_e2e_pairwise_pilot as runner
runner = importlib.reload(runner)
if RUN_STAGE == 'screening':
    branch_runtime = runner.run_branches(
        ROOT, DATASET_ROOT, GATE_A_SUMMARY, gpu_ids=GPU_IDS,
        methods=('resource','pure_sw'),
    )
    display(branch_runtime)
else:
    branch_runtime, diagnostic_runtime = runner.run_completion(
        ROOT, DATASET_ROOT, GATE_A_SUMMARY, gpu_ids=GPU_IDS
    )
    display(branch_runtime)
    display(diagnostic_runtime)

## Offline diagnostics at epoch 10/50/100

Gradient oracle is computed only after training and never affects a policy or update. Seven read-only states run on two GPUs.

In [ ]:
if RUN_STAGE == 'complete':
    print('All seven diagnostics completed by the optimized completion scheduler.')
else:
    print('Screening stage: full trajectory diagnostics are intentionally deferred.')

## Finalize the development-only result

In [ ]:
decision = pilot.finalize_screening(ROOT) if RUN_STAGE == 'screening' else pilot.finalize(ROOT)
print(json.dumps(decision, indent=2))
import pandas as pd
from IPython.display import display
if RUN_STAGE == 'screening':
    display(pd.read_csv(ROOT/'screening_method_summary.csv'))
else:
    display(pd.read_csv(ROOT/'method_summary.csv'))
    display(pd.read_csv(ROOT/'trajectory_variance_diagnostics.csv'))
assert decision['confirmatory_claim_authorized'] is False

## Export the resumable v2 screening or complete pilot

In [ ]:
required = ['common_warmup/epoch_010.pt']
if RUN_STAGE == 'screening':
    required += [
      'resource/checkpoints/epoch_050.pt','resource/checkpoints/epoch_100.pt',
      'pure_sw/checkpoints/epoch_050.pt','pure_sw/checkpoints/epoch_100.pt',
      'resource/data_subset_audit.json','pure_sw/data_subset_audit.json',
      'resource/training_provenance.json','pure_sw/training_provenance.json',
      'dense_metrics_screening.csv','screening_method_summary.csv','screening_summary.json',
    ]
else:
    required += [
      'uniform/checkpoints/epoch_050.pt','uniform/checkpoints/epoch_100.pt',
      'resource/checkpoints/epoch_050.pt','resource/checkpoints/epoch_100.pt',
      'pure_sw/checkpoints/epoch_050.pt','pure_sw/checkpoints/epoch_100.pt',
      'uniform/data_subset_audit.json','resource/data_subset_audit.json','pure_sw/data_subset_audit.json',
      'uniform/training_provenance.json','resource/training_provenance.json','pure_sw/training_provenance.json',
      'dense_metrics_all_methods.csv','method_summary.csv',
      'trajectory_variance_diagnostics.csv','summary.json',
    ]
missing = [name for name in required if not (ROOT/name).is_file() or (ROOT/name).stat().st_size == 0]
assert not missing, f'Missing final artifacts: {missing}'
bundle_name = 'rq2-e2e-pairwise-screening-v2.zip' if RUN_STAGE == 'screening' else 'rq2-e2e-pairwise-pilot-v2.zip'
bundle = Path('/kaggle/working')/bundle_name
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
    for path in ROOT.rglob('*'):
        if path.is_file(): archive.write(path, Path('e2e_pairwise_pilot_v2')/path.relative_to(ROOT))
print('Download/persist:', bundle, f'{bundle.stat().st_size/2**30:.2f} GiB')
bundle